# Setup



Avviare da terminale:

In [ ]:
conda activate llamacpp
llama-server -hf lmstudio-community/InternVL3_5-2B-GGUF:Q8_0 --ctx-size 8192  -ngl 999 --port 1234 &

SyntaxError: invalid decimal literal (2038710340.py, line 2)

In [ ]:
# Setup per esecuzione locale
import os
import sys
from pathlib import Path

# Configurazione percorsi locali
WORKSPACE_DIR = Path.cwd() / "assets" / "videos_and_frames"  # Directory corrente come workspace
VIDEO_DIR = WORKSPACE_DIR / "videos"  # Directory per i video
FRAMES_DIR = WORKSPACE_DIR / "frames_full"  # Directory per i frame estratti
SELECTED_DIR = WORKSPACE_DIR / "frames_selected"  # Directory per i frame selezionati
OUTPUT_DIR = WORKSPACE_DIR / "output"  # Directory per i risultati

# Crea le directory se non esistono
for dir_path in [VIDEO_DIR, FRAMES_DIR, SELECTED_DIR, OUTPUT_DIR]:
    dir_path.mkdir(exist_ok=True)

print(f"Workspace configurato in: {WORKSPACE_DIR}")
print(f"Directory video: {VIDEO_DIR}")
print(f"Directory frame: {FRAMES_DIR}")
print(f"Directory selezionati: {SELECTED_DIR}")
print(f"Directory output: {OUTPUT_DIR}")

Workspace configurato in: /home/gmonni54/agenticvad
Directory video: /home/gmonni54/agenticvad/videos
Directory frame: /home/gmonni54/agenticvad/frames_full
Directory selezionati: /home/gmonni54/agenticvad/frames_selected
Directory output: /home/gmonni54/agenticvad/output


Inserire il video che si vuole analizzare

In [6]:
import shutil
import glob

def list_video_files():
    """Lista i file video disponibili nella directory videos"""
    video_extensions = ['*.mp4', '*.avi', '*.mov', '*.mkv', '*.wmv', '*.flv']
    video_files = []
    
    for ext in video_extensions:
        video_files.extend(glob.glob(str(VIDEO_DIR / ext)))
        video_files.extend(glob.glob(str(VIDEO_DIR / ext.upper())))
    
    return sorted(video_files)

def select_video_interactive():
    """Selezione interattiva del video"""
    print("=== SELEZIONE VIDEO ===")
    print(f"Directory video: {VIDEO_DIR}")
    print()
    
    # Cerca video nella directory videos
    video_files = list_video_files()
    
    if not video_files:
        print("Nessun video trovato nella directory videos/")
        print()
        print("OPZIONI:")
        print("1. Copia il tuo video nella directory:", VIDEO_DIR)
        print("2. Oppure inserisci il percorso completo del video qui sotto")
        print()
        
        # Input manuale del percorso
        manual_path = input("Inserisci il percorso completo del video (o premi Invio per uscire): ").strip()
        
        if not manual_path:
            print("Nessun video selezionato. Uscita.")
            return None
            
        if not os.path.exists(manual_path):
            print(f"File non trovato: {manual_path}")
            return None
            
        return manual_path
    
    # Mostra video disponibili
    print(f"Video trovati ({len(video_files)}):")
    for i, video_file in enumerate(video_files, 1):
        file_size = os.path.getsize(video_file) / (1024**3)  # GB
        filename = Path(video_file).name
        print(f"  {i}. {filename} ({file_size:.2f} GB)")
    
    print()
    print("OPZIONI:")
    print("- Inserisci il numero del video (1, 2, 3, ...)")
    print("- Oppure inserisci il percorso completo di un altro video")
    print("- Oppure premi Invio per uscire")
    print()
    
    choice = input("Scelta: ").strip()
    
    if not choice:
        print("Nessun video selezionato. Uscita.")
        return None
    
    # Verifica se è un numero (selezione dalla lista)
    try:
        index = int(choice) - 1
        if 0 <= index < len(video_files):
            return video_files[index]
        else:
            print(f"Numero non valido. Scegli tra 1 e {len(video_files)}")
            return None
    except ValueError:
        # Non è un numero, potrebbe essere un percorso
        if os.path.exists(choice):
            return choice
        else:
            print(f"File non trovato: {choice}")
            return None

# Selezione del video
video_path = select_video_interactive()

if not video_path:
    print("Nessun video selezionato. Uscita.")
    sys.exit(1)

# Copia il video nella directory workspace se non è già lì
video_filename = Path(video_path).name
dest_path = VIDEO_DIR / video_filename

if Path(video_path) != dest_path:
    print(f"Copiando {video_filename} nella directory workspace...")
    shutil.copy2(video_path, dest_path)
    video_path = str(dest_path)

print()
print("Video selezionato con successo!")
print(f"Nome: {video_filename}")
print(f"Percorso: {video_path}")

# Verifica che il file esista e sia leggibile
if not os.path.exists(video_path):
    print(f"Errore: File non trovato: {video_path}")
    sys.exit(1)

file_size = os.path.getsize(video_path) / (1024**3)  # Dimensione in GB
print(f"Dimensione: {file_size:.2f} GB")
print()

=== SELEZIONE VIDEO ===
Directory video: /home/gmonni54/agenticvad/videos

Video trovati (1):
  1. Explosion017_x264.mp4 (0.01 GB)

OPZIONI:
- Inserisci il numero del video (1, 2, 3, ...)
- Oppure inserisci il percorso completo di un altro video
- Oppure premi Invio per uscire


Video selezionato con successo!
Nome: Explosion017_x264.mp4
Percorso: /home/gmonni54/agenticvad/videos/Explosion017_x264.mp4
Dimensione: 0.01 GB



Divido il video in frame

In [7]:
import os
import shutil
from pathlib import Path
import cv2
from PIL import Image
import numpy as np

# Usa i percorsi configurati nel setup iniziale
frames_dir = FRAMES_DIR
selected_dir = SELECTED_DIR

# Pulisci le directory esistenti
for folder in (frames_dir, selected_dir):
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

print(f"Directory frame configurate:")
print(f"  Frame completi: {frames_dir}")
print(f"  Frame selezionati: {selected_dir}")

Directory frame configurate:
  Frame completi: /home/gmonni54/agenticvad/frames_full
  Frame selezionati: /home/gmonni54/agenticvad/frames_selected


In [8]:
def extract_frames(video_path, output_dir, resize=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Impossibile aprire il video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0  # Provo a leggere gli FPS e uso 30 come fallback
    frame_index = 0
    timestamps = []

    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        if resize is not None:
            frame_rgb = cv2.resize(frame_rgb, resize)

        image = Image.fromarray(frame_rgb)
        frame_name = output_dir / f"frame_{frame_index:06d}.png"
        image.save(frame_name)

        timestamp = frame_index / fps
        timestamps.append(timestamp)

        frame_index += 1

    cap.release()
    return fps, np.array(timestamps)  # Ritorno gli FPS e l’array dei timestamp

In [12]:
video_fps, frame_timestamps = extract_frames(video_path, frames_dir, resize=None)  # Estraggo i frame (lascia resize=None per mantenere la risoluzione originale)
print(f"Frame estratti: {len(frame_timestamps)}")
print(f"FPS stimati: {video_fps:.2f}")
print(f"Durata video: {len(frame_timestamps) / video_fps:.2f} secondi")

Frame estratti: 1643
FPS stimati: 30.00
Durata video: 54.77 secondi


In [10]:
#Test con selezione dei frame per differenza
def select_frames(frames_dir, timestamps, output_dir, stride=30, diff_threshold=30.0):
    frame_paths = sorted(frames_dir.glob("*.png"))
    selected_entries = []
    previous_gray = None

    for idx, (frame_path, timestamp) in enumerate(zip(frame_paths, timestamps)):
        frame_bgr = cv2.imread(str(frame_path))
        frame_gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

        if idx % stride == 0:
            selected_entries.append((frame_path, float(timestamp)))
        elif previous_gray is not None:
            diff = cv2.absdiff(frame_gray, previous_gray)
            mean_diff = float(diff.mean())
            if mean_diff > diff_threshold:
                selected_entries.append((frame_path, float(timestamp)))

        previous_gray = frame_gray

    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for frame_path, _ in selected_entries:
        destination = output_dir / frame_path.name
        shutil.copy(frame_path, destination)

    return selected_entries

In [13]:
# Selezione a 2 fps (nessun extra per differenza): usa stride = round(fps / 2)
import math

stride_2fps = max(1, int(round(video_fps / 2.0)))
print(f"Video FPS ≈ {video_fps:.2f} -> stride per 2 fps = {stride_2fps}")

# Disattivo la selezione "per differenza" impostando una soglia enorme, così restano esattamente 2 fps
selected_frames = select_frames(
    frames_dir,
    frame_timestamps,
    selected_dir,
    stride=stride_2fps,
    diff_threshold=1e18
)
print(f"Frame selezionati (2 fps): {len(selected_frames)}")

Video FPS ≈ 30.00 -> stride per 2 fps = 15
Frame selezionati (2 fps): 110


Avvio server llamacpp

In [ ]:
#llama-server -hf lmstudio-community/InternVL3_5-2B-GGUF:Q8_0 --ctx-size 4096 --batch-size 512 --verbose --cont-batching -ngl 999 --port 1234 &> /server.log &

#!sleep 5  # Aspetta 5s per init (modello già cached, parte veloce)

# Verifica processo running
#!ps aux | grep llama-server | grep -v grep
#!lsof -i :1234

/bin/bash: /server.log: Permesso negato
Server avviato in background. Log: /content/server.log
gmonni54 2121206  0.8  0.0      0     0 ?        Z    ott02  93:13 [llama-server] <defunct>
gmonni54 3023673 97.5  1.3 101155124 1838160 pts/10 Rl 09:58  54:43 llama-server --port 8080 -hf lmstudio-community/InternVL3_5-8B-GGUF:Q8_0 -ngl 999 -np 10 --cont-batching -b 8192 -ub 2048 -c 81920
gmonni54 3040393  0.8  0.0      0     0 ?        Zs   10:49   0:02 [llama-server] <defunct>


In [14]:
import time
import requests

base_url = "http://localhost:1234"

print("Check health (aspetta 1-2 min per init GPU se prima volta)...")
deadline = time.time() + 120  # 2 min timeout
ready = False
dots = ""

while time.time() < deadline:
    try:
        r = requests.get(f"{base_url}/health", timeout=3)
        if r.status_code == 200:
            ready = True
            print("\nServer pronto su", base_url)
            break
    except requests.exceptions.ConnectionError:
        # Server non ancora pronto (normale durante init)
        dots += "."
        if len(dots) > 50:
            dots = "..."
        print(dots, end="", flush=True)
    except Exception as e:
        print(f"\nErrore: {e}")
    time.sleep(3)

if not ready:
    print("\nNon pronto. Controlla:")
    print("1. Processo running? !ps aux | grep llama-server | grep -v grep")
    print("2. Log per errori: !cat /content/server.log | tail -30")
    print("3. Porta occupata? !lsof -i :1234")
    print("Se log mostra 'out of memory', riavvia con -ngl 35 (riduci GPU layers)")
else:
    # Test testo (conferma API)
    test_payload = {
        "model": "lmstudio-community/InternVL3_5-2B-GGUF:Q8_0",
        "messages": [{"role": "user", "content": "Test riavvio: sei pronto?"}],
        "stream": False, "max_tokens": 10
    }
    try:
        r = requests.post(f"{base_url}/v1/chat/completions", json=test_payload, timeout=15)
        if r.status_code == 200:
            resp = r.json()["choices"][0]["message"]["content"]
            print("Test API OK:", resp)
        else:
            print("Errore API:", r.status_code, r.text[:100])
    except Exception as e:
        print("Eccezione API:", e)

Check health (aspetta 1-2 min per init GPU se prima volta)...

Server pronto su http://localhost:1234
Test API OK: Certamente! Rispondiamo al tuo test di


In [15]:
import requests
from PIL import Image
import base64
import io
import json
import re
from pathlib import Path

base_url = "http://localhost:1234"

def image_to_data_url(img):
    if isinstance(img, str):
        img = Image.open(img).convert("RGB")
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{b64}"

ANOMALY_PROMPT = """
You are a highly trained security analyst, reviewing footage from a surveillance camera.
You will receive a sequence of frame from a video.
Your task is to determine whether there is a clear and serious anomalous activity in the video that would require an immediate security response.


Strict Definition of Anomalous Activity:

An activity is considered anomalous only if it is an immediate security threat or highly unusual in a dangerous way.
This includes but is not limited to physical violence, explosions or fires, severe accidents or injuries, threatening or dangerous behavior.
        
Do NOT consider as anomalies normal daily activities (e.g., sitting, talking, walking, opening doors) unless there is clear evidence of danger.

        
Your response format:
        
- Short Summary: Describe what happens in the frames concisely.
        
- Anomaly Assessment; Clearly state whether the activity is truly an anomaly based on the strict definition above.
        
- Anomaly Score: Assign a value from 0 (completely normal) to 1 (clearly dangerous); only assign above 0.5 if the event is truly alarming.
        
- JSON object:
        
```json
        
{
        
"anomaly_score": <value between 0 and 1>,
        
"description": "<your justification>"
        
}
        
```

        
Here are the frames you have to analyze:
        
<frames>
""".strip()

def parse_json_any(text):
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def vision_chat(images, prompt, model="lmstudio-community/InternVL3_5-8B-GGUF:Q8_0",
                max_tokens=512, temperature=0.5, top_p=0.9):  # Definisco la chiamata chat+vision stile OpenAI
    contents = [{"type": "text", "text": prompt}]
    for img in images:
        contents.append({"type": "image_url", "image_url": {"url": image_to_data_url(img)}})  # Allego ciascuna immagine
    payload = {  # Costruisco il payload JSON per /v1/chat/completions
        "model": model,
        "messages": [{"role": "user", "content": contents}],
        "stream": False,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
    }
    r = requests.post(f"{base_url}/v1/chat/completions", json=payload, timeout=180)  # Invio la richiesta al server
    r.raise_for_status()
    data = r.json()
    return (data.get("choices") or [{}])[0].get("message", {}).get("content", "")  # Estraggo il testo della risposta

In [16]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import re

# Parametri finestra (batch)
WINDOW_SIZE = 6       # quanti frame per chiamata al modello
WINDOW_STEP = 6       # di quanto avanza la finestra (= WINDOW_SIZE per nessun overlap)
THRESHOLD = 0.5

# Dati base
frame_paths = sorted(Path(selected_dir).glob("*.png"))
print("Primi 12 selezionati:", [p.name for p in frame_paths[:12]])

# Ricavo i timestamp dei frame selezionati mantenendo l'ordine in selected_dir
# (select_frames ha restituito percorsi in frames_dir; qui ricostruisco la mappa nome->timestamp)
name_to_ts = {Path(p).name: float(ts) for (p, ts) in selected_frames}
selected_ts_sorted = [name_to_ts[p.name] for p in frame_paths]

# Preparazione timeline per viewer e modello
records = []
for b in range(0, len(frame_paths) - (WINDOW_SIZE - 1), WINDOW_STEP):
    window_paths = frame_paths[b:b + WINDOW_SIZE]
    images = [Image.open(p).convert("RGB") for p in window_paths]

    raw = vision_chat(images, ANOMALY_PROMPT)
    data = parse_json_any(raw) or {}
    score = float(data.get("anomaly_score", 0.0))
    score = max(0.0, min(1.0, score))
    desc = str(data.get("description", "") or "").strip()

    center_idx = b + (WINDOW_SIZE // 2)
    start_name = window_paths[0].name
    center_name = window_paths[WINDOW_SIZE // 2].name
    end_name = window_paths[-1].name

    start_idx = int(re.search(r"(\d+)", start_name).group(1))
    center_idx_name = int(re.search(r"(\d+)", center_name).group(1))
    end_idx = int(re.search(r"(\d+)", end_name).group(1))

    start_ts = float(selected_ts_sorted[b])
    center_ts = float(selected_ts_sorted[center_idx])
    end_ts = float(selected_ts_sorted[b + WINDOW_SIZE - 1])

    records.append({
        "frame_names": [p.name for p in window_paths],
        "frame_name_start": start_name,
        "frame_name_center": center_name,
        "frame_name_end": end_name,
        "frame_index_start": start_idx,
        "frame_index_center": center_idx_name,
        "frame_index_end": end_idx,
        "timestamp_start": start_ts,
        "timestamp_center": center_ts,
        "timestamp_end": end_ts,
        "anomaly_score": score,
        "description": desc,
        "raw": raw,
    })

# Costruisco DataFrame
df = pd.DataFrame.from_records(records)

# Flag anomalia
df["is_anomalous"] = df["anomaly_score"] >= THRESHOLD

# Metadati globali utili al viewer
VIDEO_FPS = float(video_fps)
VIDEO_DURATION = float(frame_timestamps[-1]) if len(frame_timestamps) else 0.0
SELECTED_FPS = 1.0 / float(np.median(np.diff(selected_ts_sorted))) if len(selected_ts_sorted) > 1 else 2.0

display(df.head(8))

Primi 12 selezionati: ['frame_000000.png', 'frame_000015.png', 'frame_000030.png', 'frame_000045.png', 'frame_000060.png', 'frame_000075.png', 'frame_000090.png', 'frame_000105.png', 'frame_000120.png', 'frame_000135.png', 'frame_000150.png', 'frame_000165.png']


,frame_names,frame_name_start,frame_name_center,frame_name_end,frame_index_start,frame_index_center,frame_index_end,timestamp_start,timestamp_center,timestamp_end,anomaly_score,description,raw,is_anomalous
0,"[frame_000000.png, frame_000015.png, frame_000...",frame_000000.png,frame_000045.png,frame_000075.png,0,45,75,0.0,1.5,2.5,0.0,No clear or serious anomalous activity is obse...,"```json\n{\n ""anomaly_score"": 0.0,\n ""de...",False
1,"[frame_000090.png, frame_000105.png, frame_000...",frame_000090.png,frame_000135.png,frame_000165.png,90,135,165,3.0,4.5,5.5,0.0,No clear or serious anomalous activity is obse...,"```json\n{\n ""anomaly_score"": 0.0,\n ""descri...",False
2,"[frame_000180.png, frame_000195.png, frame_000...",frame_000180.png,frame_000225.png,frame_000255.png,180,225,255,6.0,7.5,8.5,0.0,The footage shows a normal scene at a gas stat...,"```json\n{\n ""anomaly_score"": 0.0,\n ""de...",False
3,"[frame_000270.png, frame_000285.png, frame_000...",frame_000270.png,frame_000315.png,frame_000345.png,270,315,345,9.0,10.5,11.5,0.0,No clear or serious anomalous activity is obse...,"```json\n{\n ""anomaly_score"": 0.0,\n ""de...",False
4,"[frame_000360.png, frame_000375.png, frame_000...",frame_000360.png,frame_000405.png,frame_000435.png,360,405,435,12.0,13.5,14.5,0.0,No clear and serious anomalous activity is obs...,"```json\n{\n ""anomaly_score"": 0.0,\n ""de...",False
5,"[frame_000450.png, frame_000465.png, frame_000...",frame_000450.png,frame_000495.png,frame_000525.png,450,495,525,15.0,16.5,17.5,0.8,There is a group of people gathered around a m...,"```json\n{\n ""anomaly_score"": 0.8,\n ""descri...",True
6,"[frame_000540.png, frame_000555.png, frame_000...",frame_000540.png,frame_000585.png,frame_000615.png,540,585,615,18.0,19.5,20.5,0.3,A group of people is gathered around a motorcy...,"```json\n{\n ""anomaly_score"": 0.3,\n ""de...",False
7,"[frame_000630.png, frame_000645.png, frame_000...",frame_000630.png,frame_000675.png,frame_000705.png,630,675,705,21.0,22.5,23.5,0.5,There is no clear and serious anomalous activi...,"```json\n{\n ""anomaly_score"": 0.5,\n ""descri...",True


Segmenti anomali

In [17]:
segments = []
above = df["is_anomalous"].values
frames_centers = df["frame_index_center"].values

if above.any():
    start = None
    for i, flag in enumerate(above):
        if flag and start is None:
            start = i
        if (not flag or i == len(above)-1) and start is not None:
            end = i if (flag and i == len(above)-1) else i-1
            segments.append((int(frames_centers[start]), int(frames_centers[end])))
            start = None

print("Segmenti anomali (frame_start, frame_end):", segments)  # Stampo i segmenti trovati

Segmenti anomali (frame_start, frame_end): [(495, 495), (675, 675), (945, 1485)]


Visualizzazione

In [ ]:
# Viewer interattivo per ambiente locale: scorre TUTTI i frame selezionati e mostra la finestra corrispondente

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from pathlib import Path
from PIL import Image, ImageOps
import numpy as np
import textwrap, re

# Configurazione matplotlib per ambiente locale
plt.ion()  # Modalità interattiva
plt.style.use('default')  # Stile di default per compatibilità

# Dati di base
window_names = df["frame_names"].tolist()                 # finestre analizzate
frames_center = df["frame_index_center"].to_numpy()
win_start = df["timestamp_start"].to_numpy()
win_end = df["timestamp_end"].to_numpy()
scores = df["anomaly_score"].to_numpy()
descs = df["description"].fillna("").astype(str).tolist()
N_win = len(window_names)

# Tutti i frame selezionati (per scorrere come un video a 2fps)
all_names = [p.name for p in sorted(Path(selected_dir).glob("*.png"))]
S = len(all_names)
_sf = globals().get("selected_frames", [])
try:
    name_to_ts = {Path(p).name: float(ts) for (p, ts) in _sf}
except Exception:
    name_to_ts = {}
video_total = float(globals().get("VIDEO_DURATION", 0.0))
WINDOW_STEP_VAL = int(globals().get("WINDOW_STEP", 6))
WINDOW_SIZE_VAL = int(globals().get("WINDOW_SIZE", 6))

# Controlli minimi
play = widgets.Play(interval=400, value=0, min=0, max=max(0, S-1), step=1)
slider = widgets.IntSlider(value=0, min=0, max=max(0, S-1), step=1, description="Frame", continuous_update=False)
fps = widgets.FloatSlider(value=2.5, min=0.5, max=20.0, step=0.5, description="AutoPlay FPS")
widgets.jslink((play, "value"), (slider, "value"))
out = widgets.Output()

# Parametri di dimensione per massimizzare i frame
GRID_COLS = 6          # 6 colonne in sequenza
GRID_TILE = 480        # dimensione (px) di ogni riquadro
FIG_W_IN = 24          # larghezza figura in pollici
FIG_H_IN = 13          # altezza figura in pollici
FIG_DPI = 120          # DPI per aumentare i pixel effettivi


def _set_interval(change):
    play.interval = int(1000.0 / max(0.1, change["new"]))
fps.observe(_set_interval, names="value")
_set_interval({"new": fps.value})


def _wrap(text, max_chars=220, line_w=44, max_sent=2):
    parts = re.split(r"(?<=[.!?])\s+", (text or "").strip())
    head = " ".join(parts[:max_sent]) if parts else ""
    short = textwrap.shorten(head, width=max_chars, placeholder="…")
    return textwrap.fill(short, width=line_w, break_long_words=False, break_on_hyphens=False)

# Griglia immagini della finestra (parametri fissi per chiarezza)
from math import ceil

def _grid_image(paths, cols=3, tile=320):
    imgs = [Image.open(Path(selected_dir)/p).convert("RGB") for p in paths]
    if not imgs:
        return Image.new("RGB", (200, 200), (240, 240, 240))
    cols = max(1, min(cols, len(imgs)))
    rows = ceil(len(imgs) / cols)
    # garantisco 1 riga con 6 colonne quando possibile
    if len(imgs) == 6 and cols >= 6:
        cols, rows = 6, 1
    W, H = cols * tile, rows * tile
    grid = Image.new("RGB", (W, H), (255, 255, 255))
    for i, im in enumerate(imgs):
        r, c = divmod(i, cols)
        thumb = ImageOps.fit(im, (tile, tile), method=Image.LANCZOS)
        grid.paste(thumb, (c * tile, r * tile))
    return grid


def _window_idx_for_frame_index(frame_idx):
    if N_win == 0:
        return 0
    w = frame_idx // max(1, WINDOW_STEP_VAL)
    return int(min(max(0, w), N_win - 1))


def render(frame_i):
    if S == 0 or N_win == 0:
        return
    with out:
        clear_output(wait=True)
        w_idx = _window_idx_for_frame_index(frame_i)
        win = window_names[w_idx]
        ts_sel = float(name_to_ts.get(all_names[frame_i], 0.0))

        fig = plt.figure(figsize=(FIG_W_IN, FIG_H_IN), dpi=FIG_DPI)
        gs = GridSpec(3, 2, height_ratios=[2.0, 0.6, 0.8], width_ratios=[1.0, 3.0], figure=fig)

        # Anteprima frame corrente (stile "video")
        ax_prev = fig.add_subplot(gs[0, 0]); ax_prev.set_axis_off()
        curr_path = str(Path(selected_dir) / all_names[frame_i])
        ax_prev.imshow(Image.open(curr_path).convert("RGB"))
        ax_prev.set_title(f"Frame {frame_i+1}/{S}", fontsize=12)

        # Griglia della finestra (6 frame in sequenza: 6 colonne x 1 riga)
        ax_grid = fig.add_subplot(gs[0, 1]); ax_grid.set_axis_off()
        grid = _grid_image(win, cols=int(GRID_COLS), tile=int(GRID_TILE))
        ax_grid.imshow(grid)
        ax_grid.set_title(f"Finestra: {win[0]} → {win[-1]}  •  score={scores[w_idx]:.2f}", fontsize=14)

        # Caption (centro-sinistra) — testo più grande
        ax_txt = fig.add_subplot(gs[1, 0]); ax_txt.set_axis_off()
        ax_txt.text(0.5, 0.5, _wrap(descs[w_idx], max_chars=260, line_w=48), ha="center", va="center", fontsize=14, wrap=True,
                    bbox=dict(boxstyle="round,pad=0.8", fc="white", ec="black", lw=1.6))

        # Barra posizione finestra nel video (centro-destra)
        ax_bar = fig.add_subplot(gs[1, 1])
        ax_bar.set_xlim(0, max(1e-6, video_total))
        ax_bar.set_ylim(0, 1)
        ax_bar.set_yticks([])
        ax_bar.set_xlabel("Tempo video (s)", fontsize=12)
        ax_bar.axvspan(win_start[w_idx], win_end[w_idx], color="#FFC107", alpha=.35, label="Finestra")
        ax_bar.axvline(ts_sel, color="red", lw=2, label="Frame selezionato")
        ax_bar.legend(loc="upper right")

        # Curva degli score (meno spazio verticale → caratteri più piccoli)
        ax = fig.add_subplot(gs[2, :])
        ax.plot(frames_center, scores, color="#4A76FF", lw=2, label="Anomaly score")
        ax.set_ylim(0, 1); ax.set_xlabel("Frame number", fontsize=11); ax.set_ylabel("Anomaly score", fontsize=11); ax.grid(axis="y", ls="--", alpha=.4)
        TH = float(globals().get("THRESHOLD", 0.5))
        ax.axhline(TH, color="gray", ls="--", lw=1.5)
        if "segments" in globals():
            for s0, s1 in globals()["segments"]:
                ax.axvspan(s0, s1, color="red", alpha=.18)
        ax.axvline(frames_center[w_idx], color="red", lw=2)
        try: ax.set_title(str(video_filename), fontsize=12)
        except: pass

        plt.tight_layout(rect=[0.02, 0.04, 0.995, 0.98])
        plt.show()


def _on_change(change):
    if change["name"] == "value":
        render(change["new"])
slider.observe(_on_change, names="value")

controls = widgets.HBox([play, fps])
display(controls, slider, out)
render(0)

IntSlider(value=0, continuous_update=False, description='Frame', max=109)

Output()